In [2]:
import random
import pandas as pd
import numpy as np
from pathlib import Path

# === CONFIGURATION ===
NUM_SEQUENCES = 12000  # total number of samples before splitting

# Output paths
TRAIN_PATH = Path("../../../data/training/EngineOff_training")
TEST_PATH = Path("../../../data/test/EngineOff_test")
FINAL_TEST_PATH = Path("../../../data/final test/EngineOff_final test")

# Ensure parent folders exist
TRAIN_PATH.parent.mkdir(parents=True, exist_ok=True)
TEST_PATH.parent.mkdir(parents=True, exist_ok=True)
FINAL_TEST_PATH.parent.mkdir(parents=True, exist_ok=True)

# Feature names
FEATURE_COLS = ["Temperature", "Pressure", "RPM", "Vibration"]

# === DATA GENERATION ===
all_rows = []
X = []
y = []

for seq_num in range(NUM_SEQUENCES):
    state = random.choice([0, 1])
    if state == 0:
        temperature = random.uniform(-20, 49.999999)
        pressure = random.uniform(0.98, 1.02)
        rpm = 0.0
        vibration = random.uniform(0.0001, 0.001)
        label_str = "Engine Off (cold)"
    else:
        temperature = random.uniform(50, 120)
        pressure = random.uniform(0.98, 1.02)
        rpm = 0.0
        vibration = random.uniform(0.002, 0.005) if temperature >= 95 else random.uniform(0.0001, 0.001)
        label_str = "Engine Off (warm)"

    all_rows.append({
        "Temperature": temperature,
        "Pressure": pressure,
        "RPM": rpm,
        "Vibration": vibration,
        "State": label_str
    })

    X.append([[temperature, pressure, rpm, vibration]])  # shape (N, 1, 4)
    y.append(label_str)

# Convert to DataFrame and NumPy arrays
df = pd.DataFrame(all_rows)
X = np.array(X, dtype=np.float32)
y = np.array(y)

# === SPLIT INTO TRAIN / TEST / FINAL TEST ===
train_size = 8000
test_size = 2000
final_test_size = 2000

X_tr, y_tr = X[:train_size], y[:train_size]
X_te, y_te = X[train_size:train_size+test_size], y[train_size:train_size+test_size]
X_ft, y_ft = X[-final_test_size:], y[-final_test_size:]

df_tr = df.iloc[:train_size].copy()
df_te = df.iloc[train_size:train_size+test_size].copy()
df_ft = df.iloc[-final_test_size:].copy()

# === ADD Time & Sequence COLUMNS (for CSV) ===
df_tr.insert(0, "Time", 1)
df_tr.insert(1, "Sequence", np.arange(1, len(df_tr) + 1, dtype=int))
df_te.insert(0, "Time", 1)
df_te.insert(1, "Sequence", np.arange(1, len(df_te) + 1, dtype=int))
df_ft.insert(0, "Time", 1)
df_ft.insert(1, "Sequence", np.arange(1, len(df_ft) + 1, dtype=int))

# === SAVE CSV FILES ===
df_tr.to_csv(TRAIN_PATH.with_suffix(".csv"), index=False)
df_te.to_csv(TEST_PATH.with_suffix(".csv"), index=False)
df_ft.to_csv(FINAL_TEST_PATH.with_suffix(".csv"), index=False)

# === SAVE NUMPY FILES ===
np.save(TRAIN_PATH.with_name(TRAIN_PATH.name + "_X.npy"), X_tr)
np.save(TRAIN_PATH.with_name(TRAIN_PATH.name + "_y.npy"), y_tr)
np.save(TEST_PATH.with_name(TEST_PATH.name + "_X.npy"), X_te)
np.save(TEST_PATH.with_name(TEST_PATH.name + "_y.npy"), y_te)
np.save(FINAL_TEST_PATH.with_name(FINAL_TEST_PATH.name + "_X.npy"), X_ft)
np.save(FINAL_TEST_PATH.with_name(FINAL_TEST_PATH.name + "_y.npy"), y_ft)

# === PRINT SHAPES ===
print("✅ Engine Off datasets created.")
print("Training:", X_tr.shape, y_tr.shape)
print("Test:    ", X_te.shape, y_te.shape)
print("Final:   ", X_ft.shape, y_ft.shape)


✅ Engine Off datasets created.
Training: (8000, 1, 4) (8000,)
Test:     (2000, 1, 4) (2000,)
Final:    (2000, 1, 4) (2000,)
